# Tutorial: Using `hologram_pipeline.py`

This notebook explains how to use the programmatic hologram sweep pipeline in `src/scattering_calculator/simulation_pipelines/pipelines/hologram_pipeline.py`.

The pipeline is the scripted version of the interactive scattering notebooks. Instead of manually running one section at a time, you provide:

- `HologramPipelineConfig`: fixed/default physical parameters;
- `HologramPipelineRanges`: parameters to vary across simulated samples;
- `HologramPipeline`: the runner that writes an HDF5 dataset.

The example script `tutorials/simulate_hologram_sweep.py` uses the same API for larger dataset generation. This tutorial keeps the default run disabled so you can inspect and adapt the setup before spending time on a simulation.


## 1. Imports and local `src/` setup

In [ ]:
import sys
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

repo_root = Path.cwd()
if (repo_root / "src").exists() and str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

try:
    pass
    %matplotlib widget
except Exception:
    pass

plt.rcParams["figure.constrained_layout.use"] = True

from fomocid import DATA_ROOT
from scattering_calculator.simulation_pipelines import Uniform, Choice
from scattering_calculator.simulation_pipelines.pipelines import (
    HologramPipeline,
    HologramPipelineConfig,
    HologramPipelineRanges,
)


## 2. Mental model: what the pipeline does

For each sample index, the runner performs the same operations as the full scattering notebook:

1. Build the X-ray source.
2. Build detector geometry and beamstop.
3. Build the material stack.
4. Generate the magnetic pattern from a `(k0, eps, target_mean)` phase-space point and clean tiny enclosed holes.
5. Generate the FTH aperture/mask and classify stripe/bubble/mixed/saturated morphology inside the OH.
6. Build illumination.
7. Propagate `CR` and `CL` through the sample.
8. Write exit waves, ideal holograms, detected holograms, masks, morphology counts, and metadata to HDF5.

`HologramPipelineConfig` is the baseline. `HologramPipelineRanges` overrides selected values or samples a different value for each run.


## 3. Define a small fixed config

The values below are intentionally small compared with `simulate_hologram_sweep.py`. They are meant for learning and debugging, not for final training data.

Most length-like quantities are in SI units, so metres rather than nanometres or micrometres. The recipe string is the exception: material thicknesses inside the recipe are in nanometres.


In [ ]:
output_folder = DATA_ROOT / "Data" / "pipeline_tutorial"
output_path = output_folder / "pipeline_tutorial_demo.h5"

config = HologramPipelineConfig(
    recipe="Au(80)/Cr(5)/SiN(80)/Pt(4)Co(6)/Pt(2)",
    sample_name="pipeline_tutorial_demo",

    # X-ray source
    xray_energy=778.0,
    xray_photon_flux=5e8,
    xray_coherence_length=(25e-6, 25e-6),

    # Detector geometry and detector model
    detector_shape=(96, 96),
    detector_pixel_size=20e-6,
    detector_distance=0.075,
    detector_center=(48, 48),
    # Detector response only. Use readout_noise_sigma, not the legacy noise_rms alias.
    detector_params={
        "readout_noise_average": 20,
        "readout_noise_sigma": 3,
        "detector_threshold": 5e4,
        # Single canonical conversion between detector counts and photon events.
        "counts_per_photon": 100,
        "quantum_efficiency": 0.9,
    },
    # Acquisition timing/frame settings are kept separate from detector response.
    measurement_config={
        "number_frames": 1,
        "max_counts_per_image": 5e4,
        "exposure_time": 1.0,
    },

    # Beamstop
    beamstop_method="circular",
    beamstop_distance=0.010,
    beamstop_config={
        "radius": 180e-6,
        "sigma": 10e-6,
        "wire_width": 40e-6,
        "wire_bend": 30e-6,
        "angle": np.deg2rad(25),
        "antialias": 3,
        "seed": 4,
    },
    save_detected_hologram_without_beamstop=True,

    # FTH aperture. OH = object hole, RH = reference hole.
    aperture_method="FTH_circular",
    aperture_types=["OH", "RH", "RH"],
    aperture_radii=[555e-9, 100e-9, 34e-9],
    aperture_centers=[(0.0, 0.0), (-1240e-9, -1225e-9), (1225e-9, -1200e-9)],
    aperture_sigmas=[4e-9, 2e-9, 2e-9],
    aperture_angles=[0.0, 0.0, 0.0],
    aperture_ellipticities=[1.0, 1.0, 1.0],
    aperture_roughnesses=[0.0, 0.02, 0.02],
    aperture_roughness_modes=[(0, 0), (3, 10), (3, 10)],
    aperture_seeds=[1, 2, 3],
    aperture_top_radius_factors=[1.3, 1.5, 1.75],

    # Illumination
    illumination_function="gaussian",
    illumination_center=(0.0, 0.0),
    illumination_focus_distance=1e-3,
    illumination_fwhm=0.45e-6,
    illumination_alpha_beam=(0.0, 0.0),  # rad (alpha_y, alpha_x)

    # Magnetic pattern: one fixed point in the production phase space.
    pattern_type="binary_labyrinth_pattern",
    pattern_config={
        "stripe_width": 80e-9, "sigma": 6e-9,
        "H": 100, "W": 100, "n_steps": 60,
        "region": "custom", "use_gpu": False,
        "k0": 0.7, "eps": 0.8, "target_mean": 0.0,
        "noise_amp": 0.0, "quadratic_coefficient": 0.0,
        "max_hole_area": 9, "saturation_fraction_threshold": 0.01,
    },
    magnetic_pattern_classification_min_area=9,
    magnetic_pattern_bubble_max_eccentricity=0.85,
    magnetic_pattern_bubble_min_circularity=0.45,

    # Speed/memory controls
    use_roi=True,
    magnetic_pattern_use_roi=True,
    dielectric_tensor_use_roi=True,
    dielectric_tensor_compact=True,
    propagator_method="Jones",  # Use "Scalar" for the fast scalar eigenmode approximation when polarization mixing is negligible.
    scalar_refractive_index_lazy=True,  # Scalar: build refractive-index ROI patches layer by layer to reduce memory.
    propagate=False,
    jones_apply_zero_order_phase=True,  # Jones/Scalar no-FFT modes keep exp(-1j*k0*dz) between slices while skipping transverse diffraction.
    multislice_propagation_roi=True,
    # ROI padding gives local diffraction corrections room to taper smoothly back to the exp(-1j*k0*dz) baseline.
    multislice_propagation_roi_padding_px=12,
    # Overlapping padded ROI crops are merged so close apertures use one local FFT crop.
    multislice_propagation_roi_merge_overlaps=True,
    propagation_padding_px=0,
    oversampling=2,
    random_seed=0,
)


## 4. Define ranges for a sweep

`HologramPipelineRanges` tells the pipeline which values should change per sample.

Accepted patterns:

- `None`: keep the value from `config`;
- scalar value: override with a fixed value;
- `Uniform(low, high)`: sample a continuous value;
- `Choice((a, b, c))`: choose from discrete values;
- callable: compute a value from parameters already sampled for this run.

Nested dictionaries such as `pattern_config`, `beamstop_config`, `measurement_config`, and `aperture_config` can also contain samplers. The production magnetic sweep keeps `pattern_type="binary_labyrinth_pattern"` and samples `k0` from 0.1–1.1, `eps` from 0.1–1.2, and `target_mean` from -1–1; stripe/bubble/saturated state is measured after generation rather than chosen beforehand.


In [ ]:
def sampled_pattern_config(params):
    """Sample one continuous binary-domain phase-space point."""
    stripe_width = Uniform(40e-9, 120e-9).sample()
    return {
        "stripe_width": stripe_width,
        "sigma": Uniform(2e-9, 0.12 * stripe_width).sample(),
        "n_steps": 60,
        "region": "custom",
        "seed": int(np.random.randint(0, 2**31 - 1)),
        "use_gpu": False,
        "k0": Uniform(0.1, 1.1).sample(),
        "eps": Uniform(0.1, 1.2).sample(),
        "target_mean": Uniform(-1.0, 1.0).sample(),
        "noise_amp": 0.0,
        "quadratic_coefficient": 0.0,
        "max_hole_area": 9,
        "saturation_fraction_threshold": 0.01,
    }


ranges = HologramPipelineRanges(
    # Keep most parameters fixed, but demonstrate common sweep patterns.
    xray_energy=Uniform(776.0, 782.0),
    xray_coherence_length=lambda params: (
        Uniform(15e-6, 35e-6).sample(),
        Uniform(15e-6, 35e-6).sample(),
    ),
    illumination_center=(
        Uniform(-0.20e-6, 0.20e-6),
        Uniform(-0.20e-6, 0.20e-6),
    ),
    illumination_fwhm=Uniform(0.35e-6, 0.70e-6),
    # illumination_alpha_beam=(Uniform(-np.deg2rad(2), np.deg2rad(2)), Uniform(-np.deg2rad(2), np.deg2rad(2))),
    measurement_config=lambda params: {
        "number_frames": int(np.random.randint(1, 4)),
        "max_counts_per_image": Uniform(4e4, 6e4).sample(),
        "exposure_time": 1.0,
    },
    pattern_type=None,  # keep config.pattern_type = binary_labyrinth_pattern
    pattern_config=sampled_pattern_config,
    beamstop_config={
        "radius": Uniform(150e-6, 230e-6),
        "sigma": 10e-6,
        "wire_width": Uniform(30e-6, 50e-6),
        "wire_bend": Uniform(0.0, 50e-6),
        "angle": Uniform(0.0, np.pi),
        "antialias": 3,
    },
)


## 5. Run the pipeline

Running even a tiny physical simulation can take time. Keep `RUN_PIPELINE = False` while reading the notebook. Set it to `True` when you want to create `output_path`.

For a larger dataset, use the same pattern as `tutorials/simulate_hologram_sweep.py`: increase `n_samples`, use larger detector shapes, and broaden the ranges.


In [ ]:
RUN_PIPELINE = False
n_samples = 1

if RUN_PIPELINE:
    output_folder.mkdir(parents=True, exist_ok=True)
    pipeline = HologramPipeline(
        config=config,
        ranges=ranges,
        output_path=output_path,
        n_samples=n_samples,
        verbose=True,
    )
    pipeline.run()
    print(f"Wrote {output_path}")
else:
    print("Pipeline run is disabled. Set RUN_PIPELINE = True to generate the demo HDF5 file.")
    print(f"Demo output path would be: {output_path}")


## 6. Use an existing output file

If you have already run `tutorials/simulate_hologram_sweep.py`, point `output_path` at that file instead:

```python
output_path = DATA_ROOT / "Data" / "hologram_sweep" / "simulation_sweep.h5"
```

The cells below work with either the tiny demo output or a larger sweep output.


In [ ]:
# Uncomment this line to inspect the larger sweep output from simulate_hologram_sweep.py.
# output_path = DATA_ROOT / "Data" / "hologram_sweep" / "simulation_sweep.h5"

print(output_path)
print("exists:", Path(output_path).exists())


## 7. Inspect the HDF5 tree

The file keeps global file settings separate from each sample's effective simulation settings:

```text
output.h5
├── _pipeline_config/
│   ├── n_samples
│   └── oversampling
├── 00000/
│   ├── CR/                         # exit_wave, ideal, detected, optional detected_no_beamstop
│   ├── CL/                         # same datasets for the other helicity
│   ├── beamstop_mask
│   ├── supportmask
│   ├── magnetic_pattern_oh
│   └── metadata/
│       ├── sample/
│       │   ├── recipe, sample_name, real_space_pixel_size
│       │   ├── use_roi
│       │   ├── aperture/aperture_config/
│       │   ├── magnetic_pattern/        # phase point, state, bubble/stripe counts
│       │   └── dielectric_tensor/
│       ├── xray/
│       ├── detector/
│       ├── detector_params/
│       ├── measurement_config/
│       ├── artifacts_config/
│       ├── beamstop/
│       ├── illumination/
│       └── propagator_config/
│           ├── propagator_method   # written first
│           ├── propagate
│           ├── jones_apply_zero_order_phase
│           └── ...
├── 00001/
└── ...
```

Every numbered sample is self-contained. Its metadata stores the effective values actually used after sweep sampling. Sweepable settings are not repeated in `_pipeline_config`.

Important ownership rules:

- `sample/` owns the master ROI flag, aperture definition, magnetic-pattern settings, OH-local `state`, `bubble_count`, `stripe_count`, and dielectric-tensor settings. A single resolved circular component counts as one bubble.
- `propagator_config/` contains both the propagation method and every propagation option. There is no separate `propagation/` group. With `propagate=False`, `jones_apply_zero_order_phase=True` keeps the rank-zero inter-layer phase while skipping transverse FFT diffraction.
- `sample/use_roi` is the master switch; `sample/magnetic_pattern/use_roi` and `sample/dielectric_tensor/use_roi` report whether those narrower optimizations were active. Their shared leaf name is intentional, not duplicated storage.
- `detector_params/counts_per_photon` is the only counts-to-photon conversion setting.
- `detector_params/readout_noise_sigma` is the canonical readout-noise width.
- `detector_params/quantum_efficiency` is the canonical detector efficiency.
- `measurement_config`, `detector_params`, and `artifacts_config` are sibling groups because they describe different stages of the acquisition model.

Older HDF5 files may contain baseline settings under `_pipeline_config` or the previous metadata hierarchy. Use each numbered sample's effective metadata values when reading either format.


In [ ]:
def print_hdf5_tree(path, max_items=80):
    path = Path(path)
    if not path.exists():
        print(f"No file found at {path}. Run the pipeline or point output_path at an existing sweep file.")
        return

    with h5py.File(path, "r") as h5:
        print("Top-level keys:", list(h5.keys()))
        printed = 0

        def visitor(name, obj):
            nonlocal printed
            if printed >= max_items:
                return
            if isinstance(obj, h5py.Dataset):
                print(f"{name:55s} shape={obj.shape} dtype={obj.dtype}")
                printed += 1

        h5.visititems(visitor)
        if printed >= max_items:
            print(f"... stopped after {max_items} datasets")


print_hdf5_tree(output_path)


## 8. Read scalar metadata safely

Metadata values are stored as HDF5 datasets under `metadata/`. Many are scalar datasets, while aperture lists are stored as arrays.


In [ ]:
def read_dataset_value(dataset):
    value = dataset[()]
    if isinstance(value, bytes):
        return value.decode()
    if isinstance(value, np.ndarray) and value.dtype.kind == "S":
        return np.array([item.decode() for item in value])
    return value


def first_sample_key(h5):
    sample_keys = sorted(k for k in h5.keys() if k != "_pipeline_config")
    if not sample_keys:
        raise ValueError("No numbered sample groups found in this file.")
    return sample_keys[0]


def print_selected_metadata(path):
    path = Path(path)
    if not path.exists():
        print(f"No file found at {path}.")
        return

    with h5py.File(path, "r") as h5:
        sample = first_sample_key(h5)
        grp = h5[sample]
        print("sample group:", sample)
        print("recipe:", read_dataset_value(grp["metadata/sample/recipe"]))

        keys = [
            "metadata/xray/energy_eV",
            "metadata/sample/real_space_pixel_size",
            "metadata/illumination/fwhm_m",
            "metadata/illumination/focus_distance_m",
            "metadata/sample/magnetic_pattern/pattern_type_method",
            "metadata/sample/magnetic_pattern/state",
            "metadata/sample/magnetic_pattern/bubble_count",
            "metadata/sample/magnetic_pattern/stripe_count",
            "metadata/detector/shape",
            "metadata/detector/save_detected_no_beamstop",
        ]
        for key in keys:
            if key in grp:
                print(f"{key}: {read_dataset_value(grp[key])}")

        aperture_group = "metadata/sample/aperture/aperture_config"
        if aperture_group in grp:
            print("\naperture metadata:")
            for name, dataset in grp[aperture_group].items():
                print(f"  {name}: {read_dataset_value(dataset)}")


print_selected_metadata(output_path)


## 9. Load arrays for one simulation

Hologram arrays usually have shape `(N_frames, Ny, Nx)`. The helper below selects one frame when a stack is present and returns the 2-D image.


In [ ]:
def first_frame(array):
    array = np.asarray(array)
    return array[0] if array.ndim == 3 else array


def load_sample_arrays(path, sample=None, frame=0):
    path = Path(path)
    if not path.exists():
        print(f"No file found at {path}.")
        return None

    with h5py.File(path, "r") as h5:
        sample = sample or first_sample_key(h5)
        grp = h5[sample]
        out = {
            "sample": sample,
            "CR_ideal": first_frame(grp["CR/ideal"][frame:frame + 1]),
            "CL_ideal": first_frame(grp["CL/ideal"][frame:frame + 1]),
            "CR_detected": first_frame(grp["CR/detected"][frame:frame + 1]),
            "CL_detected": first_frame(grp["CL/detected"][frame:frame + 1]),
            "CR_exit_wave": first_frame(grp["CR/exit_wave"][frame:frame + 1]),
            "CL_exit_wave": first_frame(grp["CL/exit_wave"][frame:frame + 1]),
            "beamstop_mask": grp["beamstop_mask"][()] if "beamstop_mask" in grp else None,
            "supportmask": grp["supportmask"][()] if "supportmask" in grp else None,
            "magnetic_pattern_oh": grp["magnetic_pattern_oh"][()] if "magnetic_pattern_oh" in grp else None,
        }
        if "CR/detected_no_beamstop" in grp:
            out["CR_detected_no_beamstop"] = first_frame(grp["CR/detected_no_beamstop"][frame:frame + 1])
            out["CL_detected_no_beamstop"] = first_frame(grp["CL/detected_no_beamstop"][frame:frame + 1])
        return out


arrays = load_sample_arrays(output_path)
if arrays is not None:
    for key, value in arrays.items():
        if isinstance(value, np.ndarray):
            print(f"{key:28s} shape={value.shape} dtype={value.dtype}")
        else:
            print(f"{key:28s} {value}")


## 10. Quick hologram overview

This compact view is a first sanity check: ideal and detected holograms for both helicities, plus the most important saved masks. More detailed mask, magnetic-pattern, and exit-wave views follow in the next sections.


In [ ]:
def show(ax, image, title, cmap="viridis", percentiles=(1, 99), vmin=None, vmax=None):
    if image is None:
        ax.text(0.5, 0.5, "not saved", ha="center", va="center")
        ax.set_title(title)
        ax.set_axis_off()
        return None
    data = np.asarray(image)
    if np.iscomplexobj(data):
        data = np.abs(data)
    if vmin is None or vmax is None:
        vmin, vmax = np.nanpercentile(data, percentiles)
        if np.isclose(vmin, vmax):
            vmin, vmax = float(np.nanmin(data)), float(np.nanmax(data))
            if np.isclose(vmin, vmax):
                vmin, vmax = vmin - 0.5, vmax + 0.5
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_axis_off()
    return im


if arrays is None:
    print("No arrays loaded yet.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(13, 6.5))
    panels = [
        ("CR_ideal", "CR ideal", "magma"),
        ("CL_ideal", "CL ideal", "magma"),
        ("CR_detected", "CR detected", "magma"),
        ("CL_detected", "CL detected", "magma"),
        ("beamstop_mask", "beamstop mask", "gray"),
        ("supportmask", "holography support mask", "gray"),
        ("magnetic_pattern_oh", "magnetic pattern in OH", "RdBu_r"),
        ("CR_exit_wave", "CR exit wave amplitude", "viridis"),
    ]
    for ax, (key, title, cmap) in zip(axes.flat, panels):
        if key == "magnetic_pattern_oh":
            show(ax, arrays.get(key), title, cmap=cmap, vmin=-1, vmax=1)
        else:
            show(ax, arrays.get(key), title, cmap=cmap)


## 11. Holography mask visualization

The pipeline saves two mask-like outputs per simulation:

- `beamstop_mask`: detector-plane shadow of the beamstop and its support wire.
- `supportmask`: reconstruction-grid support mask derived from the FTH object/reference holes.

The aperture geometry itself is stored in metadata. The layout panel below redraws object holes (`OH`), reference holes (`RH`), and rectangular slits (`SLIT`) from that metadata in real-space units.


In [ ]:
from matplotlib.patches import Ellipse, Rectangle


def load_aperture_metadata(path, sample=None):
    path = Path(path)
    if not path.exists():
        return None
    with h5py.File(path, "r") as h5:
        sample = sample or first_sample_key(h5)
        grp = h5[sample]
        aperture_group = "metadata/sample/aperture/aperture_config"
        if aperture_group not in grp:
            return None
        meta = {}
        for name, dataset in grp[aperture_group].items():
            meta[name] = read_dataset_value(dataset)
        return meta


def plot_aperture_layout(ax, aperture_meta):
    if aperture_meta is None:
        ax.text(0.5, 0.5, "aperture metadata not saved", ha="center", va="center")
        ax.set_axis_off()
        return

    types = aperture_meta["apertures_type"]
    radii = np.asarray(aperture_meta["apertures_radius"], dtype=float)
    centers = np.asarray(aperture_meta["apertures_center"], dtype=float)
    lengths = np.asarray(aperture_meta.get("apertures_length", np.zeros(len(radii))), dtype=float)
    angles = np.asarray(aperture_meta.get("apertures_angle", np.zeros(len(radii))), dtype=float)
    ellipticities = np.asarray(aperture_meta.get("apertures_ellipticity", np.ones(len(radii))), dtype=float)
    top_factors = np.asarray(aperture_meta.get("apertures_top_radius_factor", np.ones(len(radii))), dtype=float)

    for aperture_type, radius, length, center, angle, ellipticity, top_factor in zip(
        types, radii, lengths, centers, angles, ellipticities, top_factors
    ):
        display_radius = radius * max(1.0, float(top_factor))
        color = "tab:blue" if aperture_type == "OH" else ("tab:green" if aperture_type == "SLIT" else "tab:orange")
        if aperture_type == "SLIT":
            display_length = length * max(1.0, float(top_factor))
            patch = Rectangle(
                xy=((center[1] - 0.5 * display_length) * 1e6, (center[0] - 0.5 * display_radius) * 1e6),
                width=display_length * 1e6,
                height=display_radius * 1e6,
                angle=np.rad2deg(angle),
                rotation_point=(center[1] * 1e6, center[0] * 1e6),
                facecolor=color,
                edgecolor="black",
                alpha=0.45,
                lw=1.5,
            )
        else:
            radius_y = display_radius * np.sqrt(ellipticity)
            radius_x = display_radius / np.sqrt(ellipticity)
            patch = Ellipse(
                xy=(center[1] * 1e6, center[0] * 1e6),
                width=2 * radius_x * 1e6,
                height=2 * radius_y * 1e6,
                angle=np.rad2deg(angle),
                facecolor=color,
                edgecolor="black",
                alpha=0.45,
                lw=1.5,
            )
        ax.add_patch(patch)
        ax.text(center[1] * 1e6, center[0] * 1e6, aperture_type, ha="center", va="center")

    extents_y = np.where(np.asarray(types) == "SLIT", np.maximum(radii, lengths) * top_factors, radii * top_factors)
    extents_x = extents_y
    span_y = np.max(np.abs(centers[:, 0]) + extents_y) if len(radii) else 1e-6
    span_x = np.max(np.abs(centers[:, 1]) + extents_x) if len(radii) else 1e-6
    span = max(span_x, span_y, 1e-6) * 1.3 * 1e6
    ax.set_xlim(-span, span)
    ax.set_ylim(-span, span)
    ax.set_aspect("equal")
    ax.set_title("aperture layout from metadata")
    ax.set_xlabel("x in um")
    ax.set_ylabel("y in um")
    ax.grid(alpha=0.25)


if arrays is None:
    print("No arrays loaded yet.")
else:
    aperture_meta = load_aperture_metadata(output_path, arrays["sample"])
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    show(axes[0], arrays.get("beamstop_mask"), "detector beamstop mask", cmap="gray", vmin=0, vmax=1)
    show(axes[1], arrays.get("supportmask"), "FTH support mask", cmap="gray", vmin=0, vmax=1)
    plot_aperture_layout(axes[2], aperture_meta)


## 12. Magnetic pattern visualization

`magnetic_pattern_oh` is the magnetic texture restricted to the object-hole region that is visible to the FTH mask. This is the most useful saved magnetic-pattern target for quick dataset inspection.


In [ ]:
if arrays is None:
    print("No arrays loaded yet.")
else:
    pattern = arrays.get("magnetic_pattern_oh")
    if pattern is None:
        print("magnetic_pattern_oh was not saved.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
        show(axes[0], pattern, "magnetic pattern in object hole", cmap="RdBu_r", vmin=-1, vmax=1)
        axes[1].hist(np.asarray(pattern).ravel(), bins=80, range=(-1, 1), color="0.25")
        axes[1].set_title("magnetic-pattern histogram")
        axes[1].set_xlabel("m_z")
        axes[1].set_ylabel("pixels")
        print("magnetic_pattern_oh min/max/mean:", float(np.min(pattern)), float(np.max(pattern)), float(np.mean(pattern)))


## 13. Exit wave visualization for CR, CL, and difference

The pipeline saves complex scalar exit waves for both helicities. The rows below show `CR`, `CL`, and `CR - CL`. Columns show amplitude, phase, and a compact RGB complex-field representation where phase is hue and magnitude controls brightness.


In [ ]:
def complex_field_to_rgb(field, magnitude_clip_percentile=99):
    magnitude = np.abs(field)
    phase = np.angle(field)
    scale = np.percentile(magnitude, magnitude_clip_percentile)
    if scale <= 0:
        scale = 1.0
    normalized_magnitude = np.clip(magnitude / scale, 0, 1)
    hue = (phase + np.pi) / (2 * np.pi)
    rgb = plt.cm.hsv(hue)[..., :3]
    return rgb * normalized_magnitude[..., None] ** 2


if arrays is None:
    print("No arrays loaded yet.")
else:
    exit_waves = {
        "CR exit wave": arrays["CR_exit_wave"],
        "CL exit wave": arrays["CL_exit_wave"],
        "CR - CL exit wave": arrays["CR_exit_wave"] - arrays["CL_exit_wave"],
    }

    fig, axes = plt.subplots(len(exit_waves), 3, figsize=(11, 9))
    for row, (label, field) in enumerate(exit_waves.items()):
        amplitude = np.abs(field)
        phase = np.angle(field)
        show(axes[row, 0], amplitude, f"{label}: amplitude", cmap="viridis")
        axes[row, 1].imshow(phase, cmap="hsv", vmin=-np.pi, vmax=np.pi)
        axes[row, 1].set_title(f"{label}: phase")
        axes[row, 1].set_axis_off()
        axes[row, 2].imshow(complex_field_to_rgb(field))
        axes[row, 2].set_title(f"{label}: RGB complex field")
        axes[row, 2].set_axis_off()


## 14. Compute helicity sum, difference, and FTH reconstruction

The pipeline writes `CR` and `CL`. Magnetic contrast is commonly inspected with `CR - CL`; charge/background is commonly inspected with `CR + CL`.

The simple FTH reconstruction used in the notebooks is:

```python
fftshift(fft2(fftshift(hologram)))
```


In [ ]:
def fth_reconstruct(hologram):
    return np.fft.fftshift(np.fft.fft2(np.fft.fftshift(hologram)))


if arrays is None:
    print("No arrays loaded yet.")
else:
    ideal_diff = arrays["CR_ideal"] - arrays["CL_ideal"]
    detected_diff = arrays["CR_detected"] - arrays["CL_detected"]
    detected_sum = arrays["CR_detected"] + arrays["CL_detected"]
    reconstruction = fth_reconstruct(detected_diff)

    fig, axes = plt.subplots(2, 2, figsize=(9, 7))
    show(axes[0, 0], ideal_diff, "ideal CR - CL", cmap="RdBu_r")
    show(axes[0, 1], detected_diff, "detected CR - CL", cmap="RdBu_r")
    show(axes[1, 0], detected_sum, "detected CR + CL", cmap="magma")
    show(axes[1, 1], np.abs(reconstruction), "|FTH reconstruction of detected diff|", cmap="inferno")


## 15. Reading many samples

For dataset work, iterate over numbered groups and keep `_pipeline_config` separate. This pattern is useful for training-set loaders and quick quality-control summaries.


In [ ]:
def iter_sample_groups(h5):
    for key in sorted(k for k in h5.keys() if k != "_pipeline_config"):
        yield key, h5[key]


def summarize_file(path, max_samples=10):
    path = Path(path)
    if not path.exists():
        print(f"No file found at {path}.")
        return

    rows = []
    with h5py.File(path, "r") as h5:
        for key, grp in iter_sample_groups(h5):
            row = {
                "sample": key,
                "frames": grp["CR/detected"].shape[0],
                "detector_shape": grp["CR/detected"].shape[-2:],
                "mean_CR_detected": float(np.mean(grp["CR/detected"][0])),
                "mean_CL_detected": float(np.mean(grp["CL/detected"][0])),
            }
            if "metadata/xray/energy_eV" in grp:
                row["energy_eV"] = float(grp["metadata/xray/energy_eV"][()])
            if "metadata/sample/magnetic_pattern/pattern_type_method" in grp:
                row["pattern_type"] = read_dataset_value(grp["metadata/sample/magnetic_pattern/pattern_type_method"])
            if "metadata/sample/magnetic_pattern/state" in grp:
                row["magnetic_state"] = read_dataset_value(grp["metadata/sample/magnetic_pattern/state"])
                row["bubble_count"] = int(grp["metadata/sample/magnetic_pattern/bubble_count"][()])
                row["stripe_count"] = int(grp["metadata/sample/magnetic_pattern/stripe_count"][()])
            rows.append(row)
            if len(rows) >= max_samples:
                break

    for row in rows:
        print(row)


summarize_file(output_path)


## 16. Practical checklist

Before launching a large sweep:

- Run `n_samples=1` first and inspect the output tree.
- Verify `CR/detected`, `CL/detected`, `beamstop_mask`, `supportmask`, and `metadata`.
- Check the sampled metadata ranges are physically sensible.
- Keep `random_seed` fixed for reproducible debugging; set it to `None` for non-reproducible production sweeps.
- Use `dielectric_tensor_compact=True` for Jones memory efficiency. In Scalar mode, keep `scalar_refractive_index_lazy=True` so refractive-index ROI patches are built layer by layer during propagation.
- Increase detector size, frame count, and sweep ranges only after the tiny run looks correct.
